# F-MG-0-SUPP: Group LASSO Grouping vs. Microphone Count

Noiseless multifrequency reconstruction with 10 candidate sources and 2 active sources.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root()
sys.path[:0] = [
    str(REPO_ROOT / "src"),
    str(REPO_ROOT / "notebooks" / "benchmarks_v2" / "functions"),
]

from figures import save_pdf
from multiple_frequency_config import (
    ALL_METHOD_ORDER,
    ALPHA,
    COMPARISON_METHOD_ORDER,
    MAX_ITER,
    MIC_COUNTS,
    METHOD_COLORS,
    METHOD_LABELS,
    METHOD_LINESTYLES,
    METHOD_ORDER,
    NUM_ACTIVE,
    NUM_SOURCES,
    SEEDS,
    SOURCE_ANGLE_SPAN,
    SOURCE_SPACING_DEG,
    base_sim_kwargs,
)
from single_frequency_benchmarks import f1_threshold_fraction, plot_metric, relabel_legend
from cs_priors.simulation.mixing_model import quick_frequency_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve

TAG = "F-MG-0-SUPP"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / "v2" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SIM_KWARGS = base_sim_kwargs()


def relative_complex_error(X_hat, X_true):
    return np.linalg.norm(X_hat - X_true) / np.linalg.norm(X_true)


def make_simulation(num_mics, seed):
    return quick_frequency_sim(num_mics=int(num_mics), seed=int(seed), **SIM_KWARGS)


def run_case(num_mics, seed):
    sim = make_simulation(num_mics, seed)
    X0 = sim.X_pinv
    solutions = {
        "LASSO": frequency_lasso_solve(
            sim.Y, sim.A, alpha=ALPHA, max_iter=MAX_ITER, X_start=X0
        ),
        "Group LASSO, no groups": frequency_group_lasso_solve(
            sim.Y, sim.A, alpha=ALPHA, grouping="none", max_iter=MAX_ITER, X_start=X0
        ),
        "Group LASSO, frequency groups": frequency_group_lasso_solve(
            sim.Y, sim.A, alpha=ALPHA, grouping="frequency", max_iter=MAX_ITER, X_start=X0
        ),
        "MP": X0,
        "Sparse Prior, no groups": sparse_prior_solve(X0, sim.A, grouping="none"),
        "Sparse Prior, frequency groups": sparse_prior_solve(
            X0, sim.A, grouping="frequency"
        ),
    }
    lasso_group_none_difference = np.linalg.norm(
        solutions["Group LASSO, no groups"] - solutions["LASSO"]
    ) / max(np.linalg.norm(solutions["LASSO"]), 1e-12)

    return [
        {
            "num_mics": int(num_mics),
            "seed": int(seed),
            "num_freqs": int(sim.A.shape[2]),
            "method": method,
            "X_hat": X_hat,
            "relative_complex_error": relative_complex_error(X_hat, sim.X),
            "f1_threshold_10_percent": f1_threshold_fraction(X_hat, sim.active_indices),
            "lasso_group_none_relative_difference": lasso_group_none_difference,
        }
        for method, X_hat in solutions.items()
    ]


# Sanity check

In [ ]:
from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plot_geometry import plot_sim_geometry


CHECK_NUM_MICS = 3
CHECK_SEED = 2

check_sim = make_simulation(CHECK_NUM_MICS, CHECK_SEED)
check_rows = run_case(CHECK_NUM_MICS, CHECK_SEED)

plot_sim_geometry(check_sim, show=True)

plot_matrices(
    [check_sim.X, *[row["X_hat"] for row in check_rows]],
    ["X true", *[METHOD_LABELS.get(row["method"], row["method"]) for row in check_rows]],
    show_values=False,
)

fig = plt.gcf()
save_pdf(fig, FIGURE_DIR / "example_solution_matrices.pdf")


# Amplitude Heatmap

In [ ]:
def filename_stem(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace(",", "")
        .replace("-", "_")
    )


def save_amplitude_heatmaps(solutions, freqs, save_dir, prefix="amplitude_heatmap"):
    amplitudes = {name: np.abs(X) for name, X in solutions.items()}
    vmax = max(float(A.max()) for A in amplitudes.values())
    freq_labels = [f"{freq / 1000:g}" for freq in freqs]

    for name, amplitude in amplitudes.items():
        display_name = METHOD_LABELS.get(name, name)
        fig, ax = plt.subplots(figsize=(5.4, 4.2))
        image = ax.imshow(amplitude, aspect="auto", origin="upper", vmin=0, vmax=vmax)

        ax.set_title(display_name)
        ax.set_xlabel("Frequency component (kHz)")
        ax.set_ylabel("Source index")
        ax.set_xticks(np.arange(len(freqs)))
        ax.set_xticklabels(freq_labels, rotation=45, ha="right")
        ax.set_yticks(np.arange(amplitude.shape[0]))

        cbar = fig.colorbar(image, ax=ax)
        cbar.set_label("Amplitude")
        fig.tight_layout()

        save_pdf(fig, save_dir / "heatmap_solutions" / f"{prefix}_{filename_stem(name)}.pdf")
        # plt.show()


heatmap_solutions = {
    "X true": check_sim.X,
    **{row["method"]: row["X_hat"] for row in check_rows},
}

save_amplitude_heatmaps(heatmap_solutions, check_sim.freqs, FIGURE_DIR)


In [ ]:
def method_solution(rows, method):
    return next(row["X_hat"] for row in rows if row["method"] == method)


def phase_error(X_hat, X_true):
    return np.angle(X_hat * np.conj(X_true))


phase_methods = [
    "Group LASSO, no groups",
    "Group LASSO, frequency groups",
    "Sparse Prior, no groups",
    "Sparse Prior, frequency groups",
]

X_true = check_sim.X
source_scores = np.linalg.norm(X_true, axis=1)
source_mask = source_scores >= 0.1 * source_scores.max()
selected_sources = np.flatnonzero(source_mask)
freq_labels = [f"{freq / 1000:g}" for freq in check_sim.freqs]
phase_dir = FIGURE_DIR / "phase_difference"

for method in phase_methods:
    X_hat = method_solution(check_rows, method)
    fig, ax = plt.subplots(figsize=(5.4, 3.4), constrained_layout=True)
    image = ax.imshow(
        phase_error(X_hat, X_true)[source_mask, :],
        aspect="auto",
        origin="upper",
        cmap="coolwarm",
        vmin=-np.pi,
        vmax=np.pi,
    )
    ax.set_title(method)
    ax.set_xlabel("Frequency component (kHz)")
    ax.set_ylabel("Source index")
    ax.set_xticks(np.arange(len(freq_labels)))
    ax.set_xticklabels(freq_labels, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(selected_sources)))
    ax.set_yticklabels(selected_sources)

    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label("Wrapped phase error (rad)")

    save_pdf(fig, phase_dir / f"phase_error_{filename_stem(method)}.pdf")
    plt.show()


# Run the simulations 
(10 sources, 5 seeds 19 seconds, mics 3,4,5,6,7)

(10 sources, 5 seeds, mics 2,3,4,5,6,7: 23 seconds)

(11 sources, 5 seeds, mics 3,4,5,6,7: 22.3 seconds)

(12 sources, 5 seeds, mics 2,3,4,5,6,7: 30 seconds)

(13 sources, 5 seeds, mics 2,3,4,5,6,7, 36 seconds)

(15 sources, 5 seeds, mics 2,3,4,5,6,7, 43.2 seconds)


In [ ]:
rows = []
for num_mics in MIC_COUNTS:
    for seed in SEEDS:
        rows.extend(run_case(num_mics, seed))

results_df = pd.DataFrame(rows)
results_df["method"] = pd.Categorical(results_df["method"], categories=ALL_METHOD_ORDER, ordered=True)
results_df = results_df.sort_values(["num_mics", "seed", "method"]).reset_index(drop=True)

diagnostic_df = (
    results_df[["num_mics", "seed", "num_freqs", "lasso_group_none_relative_difference"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# diagnostic_df

`grouping="none"` gives singleton groups and should match plain LASSO up to solver/numerical differences. `grouping="frequency"` groups all frequencies for each source.

In [ ]:
fig, ax, summary_df = plot_metric(
    results_df,
    x_col="num_mics",
    method_order=METHOD_ORDER,
    colors=METHOD_COLORS,
    linestyles=METHOD_LINESTYLES,
    xlabel="Number of microphones",
    title=f"Relative error for {NUM_SOURCES} multifrequency sources, noiseless",
    xscale="linear",
    yscale="linear",
)

ax.set_xticks(MIC_COUNTS)

save_pdf(fig, FIGURE_DIR / "group_lasso_grouping_vs_num_mics.pdf")
plt.show()

# summary_df


In [ ]:
fig, ax, sparse_summary_df = plot_metric(
    results_df,
    x_col="num_mics",
    method_order=COMPARISON_METHOD_ORDER,
    colors=METHOD_COLORS,
    linestyles=METHOD_LINESTYLES,
    xlabel="Number of microphones",
    ylabel="Relative complex error",
    title=f"Relative error for {NUM_SOURCES} multifrequency sources, noiseless",
    xscale="linear",
    yscale="linear",
)

ax.set_xticks(MIC_COUNTS)
relabel_legend(ax, METHOD_LABELS)

save_pdf(fig, FIGURE_DIR / "sparse_prior_grouping_relative_error_vs_num_mics.pdf")
plt.show()

# sparse_summary_df


# F1

In [ ]:
fig, ax, f1_summary_df = plot_metric(
    results_df,
    x_col="num_mics",
    y_col="f1_threshold_10_percent",
    method_order=METHOD_ORDER,
    colors=METHOD_COLORS,
    linestyles=METHOD_LINESTYLES,
    xlabel="Number of microphones",
    ylabel=r"$F_1$-score",
    title=f"Support recovery for {NUM_SOURCES} multifrequency sources, noiseless",
    xscale="linear",
    yscale="linear",
)

ax.set_xticks(MIC_COUNTS)
ax.set_ylim(-0.02, 1.02)

save_pdf(fig, FIGURE_DIR / "f1_threshold_10_percent_vs_num_mics.pdf")
plt.show()

# f1_summary_df


In [ ]:
fig, ax, sparse_f1_summary_df = plot_metric(
    results_df,
    x_col="num_mics",
    y_col="f1_threshold_10_percent",
    method_order=COMPARISON_METHOD_ORDER,
    colors=METHOD_COLORS,
    linestyles=METHOD_LINESTYLES,
    xlabel="Number of microphones",
    ylabel=r"$F_1$-score",
    title=f"Support recovery for {NUM_SOURCES} multifrequency sources, noiseless",
    xscale="linear",
    yscale="linear",
)

ax.set_xticks(MIC_COUNTS)
relabel_legend(ax, METHOD_LABELS)
ax.set_ylim(-0.02, 1.02)

save_pdf(fig, FIGURE_DIR / "sparse_prior_grouping_f1_vs_num_mics.pdf")
plt.show()

# sparse_f1_summary_df
